# Delta table Query util
DuckDB natively supports reading Delta tables using `delta_scan()`.

In [1]:
import os
import re

import certifi
import duckdb
import pandas as pd

from electricity_maps.config import get_settings

os.environ["AWS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()
pd.set_option('display.max_columns', None)

get_settings.cache_clear()
settings = get_settings()
DATA_DIR = settings.data_dir


In [2]:
def get_sql_df(sql_query):
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs; INSTALL aws; LOAD aws;")
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET aws_s3 (TYPE S3, KEY_ID '{settings.aws_access_key_id}', SECRET '{settings.aws_secret_access_key}', REGION '{settings.aws_region}');")

    def replace_table(match):
        keyword = match.group(1)
        schema = match.group(2)
        table = match.group(3)

        return f"{keyword} delta_scan('{DATA_DIR}/{schema}/{table}')"

    parsed_query = re.sub(
        r'(?i)(FROM|JOIN)\s+([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)',
        replace_table,
        sql_query
    )
    return con.execute(parsed_query).df()

def print_sql_df(sql_query):
    df = get_sql_df(sql_query)
    return df

In [3]:
print("Bronze Daily Flows Summary:")
print_sql_df('select * from bronze.electricity_flows order by process_ts desc limit 10')

Bronze Daily Flows Summary:


,process_ts,ingestion_timestamp,source_url,zone,range_start,range_end,raw_json,record_count,year,month,day
0,1777223123493,2026-04-26 22:35:23.493984+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-26 18:30:00+05:30,2026-04-26 22:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",4,2026,4,26


In [4]:
print("Bronze Daily Mix Summary:")
print_sql_df('select * from bronze.electricity_mix order by process_ts desc limit 10')

Bronze Daily Mix Summary:


,process_ts,ingestion_timestamp,source_url,zone,range_start,range_end,raw_json,record_count,year,month,day
0,1777223123493,2026-04-26 22:35:23.493984+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-26 18:30:00+05:30,2026-04-26 22:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",4,2026,4,26


In [11]:
print("Silver :")
print_sql_df('select * from silver.electricity_mix order by process_ts desc limit 6')

Silver :


,process_ts,zone,datetime,updated_at,is_estimated,estimation_method,nuclear_mw,geothermal_mw,biomass_mw,coal_mw,wind_mw,solar_mw,hydro_mw,gas_mw,oil_mw,unknown_mw,hydro_storage_charge_mw,hydro_storage_discharge_mw,battery_storage_charge_mw,battery_storage_discharge_mw,flow_exports_mw,flow_imports_mw,year,month,day
0,1777223153884,FR,2026-04-26 16:00:00,2026-04-26 16:39:27.255,True,SANDBOX_MODE_DATA,30448.361,NaN,983.667,0.0,609.967,10484.144,3709.205,354.580,29.799,NaN,2851.521,NaN,NaN,13.048,10939.0,0.0,2026,4,26
1,1777223153884,FR,2026-04-26 14:00:00,2026-04-26 16:10:07.341,True,SANDBOX_MODE_DATA,34026.659,NaN,1420.101,0.0,514.181,10038.885,3252.378,403.698,37.449,NaN,2553.868,NaN,NaN,36.084,9376.0,0.0,2026,4,26
2,1777223153884,FR,2026-04-26 15:00:00,2026-04-26 16:39:27.255,True,SANDBOX_MODE_DATA,28114.943,NaN,1052.044,0.0,904.815,8261.493,3854.653,388.716,39.197,NaN,2278.633,NaN,NaN,58.605,7960.0,242.0,2026,4,26
3,1777223153884,FR,2026-04-26 13:00:00,2026-04-26 16:10:07.341,True,SANDBOX_MODE_DATA,23223.345,NaN,928.774,0.0,573.591,10752.647,2564.090,260.748,40.678,NaN,2132.566,NaN,NaN,55.957,10443.0,100.0,2026,4,26
4,1777212885259,FR,2026-04-26 12:00:00,2026-04-26 13:55:18.001,True,SANDBOX_MODE_DATA,23198.216,NaN,929.576,0.0,569.769,11423.678,2518.921,271.613,40.678,NaN,2303.251,NaN,12.585,NaN,9378.0,570.0,2026,4,26
5,1777206207772,FR,2026-04-26 09:00:00,2026-04-26 11:54:21.220,True,SANDBOX_MODE_DATA,24184.452,NaN,934.787,0.0,670.105,9236.720,2752.129,329.130,40.678,NaN,2092.978,NaN,6.517,NaN,8865.0,1151.0,2026,4,26


In [6]:
print("Silver :")
print_sql_df('select * from silver.electricity_flows order by process_ts desc limit 10')

Silver :


,process_ts,zone,datetime,updated_at,neighbor_zone,direction,power_mw,year,month,day
0,1777223153884,FR,2026-04-26 13:00:00,2026-04-26 16:10:07.341,BE,import,2365.0,2026,4,26
1,1777223153884,FR,2026-04-26 13:00:00,2026-04-26 16:10:07.341,ES,import,2666.0,2026,4,26
2,1777223153884,FR,2026-04-26 13:00:00,2026-04-26 16:10:07.341,DE,import,1028.0,2026,4,26
3,1777223153884,FR,2026-04-26 13:00:00,2026-04-26 16:10:07.341,GB,export,3016.0,2026,4,26
4,1777223153884,FR,2026-04-26 14:00:00,2026-04-26 16:10:07.341,BE,import,2044.0,2026,4,26
5,1777223153884,FR,2026-04-26 13:00:00,2026-04-26 16:10:07.341,CH,export,89.0,2026,4,26
6,1777223153884,FR,2026-04-26 14:00:00,2026-04-26 16:10:07.341,CH,import,602.0,2026,4,26
7,1777223153884,FR,2026-04-26 13:00:00,2026-04-26 16:10:07.341,IT-NO,export,180.0,2026,4,26
8,1777223153884,FR,2026-04-26 13:00:00,2026-04-26 16:10:07.341,LU,export,127.0,2026,4,26
9,1777223153884,FR,2026-04-26 14:00:00,2026-04-26 16:10:07.341,DE,import,1691.0,2026,4,26


In [7]:
print("Gold Daily Exports:")
print_sql_df('select * from gold.daily_exports order by process_ts desc limit 10')

Gold Daily Exports:


,process_ts,zone,zone_name,destination_zone,destination_zone_name,date,export_mwh,hours_covered,year,month,day
0,1777223204564,FR,France,GB,Great Britain,2026-04-26,52833.0,17,2026,4,26
1,1777223204564,FR,France,LU,LU,2026-04-26,3455.0,17,2026,4,26
2,1777223204564,FR,France,IT-NO,Italy North,2026-04-26,13718.0,17,2026,4,26
3,1777205541513,FR,France,IT-NO,Italy North,2026-04-25,26146.0,12,2026,4,25
4,1777205541513,FR,France,GB,Great Britain,2026-04-25,36655.0,12,2026,4,25
5,1777205541513,FR,France,LU,LU,2026-04-25,2315.0,12,2026,4,25


In [8]:
print("\nGold Daily Imports:")
print_sql_df("SELECT * FROM gold.daily_imports order by process_ts desc LIMIT 10")


Gold Daily Imports:


,process_ts,zone,zone_name,source_zone,source_zone_name,date,import_mwh,hours_covered,year,month,day
0,1777223204564,FR,France,DE,Germany,2026-04-26,25831.0,17,2026,4,26
1,1777223204564,FR,France,ES,Spain,2026-04-26,34489.0,17,2026,4,26
2,1777223204564,FR,France,CH,Switzerland,2026-04-26,8750.0,16,2026,4,26
3,1777223204564,FR,France,BE,Belgium,2026-04-26,55254.0,17,2026,4,26
4,1777205541513,FR,France,ES,Spain,2026-04-25,28389.0,12,2026,4,25
5,1777205541513,FR,France,CH,Switzerland,2026-04-25,17661.0,12,2026,4,25
6,1777205541513,FR,France,DE,Germany,2026-04-25,15160.0,12,2026,4,25
7,1777205541513,FR,France,BE,Belgium,2026-04-25,31998.0,12,2026,4,25


In [9]:
print("\nGold Daily Mix:")
print_sql_df("SELECT * FROM gold.daily_electricity_mix order by process_ts desc LIMIT 10")


Gold Daily Mix:


,process_ts,zone,zone_name,date,nuclear_pct,biomass_pct,wind_pct,solar_pct,hydro_pct,gas_pct,oil_pct,coal_pct,geothermal_pct,unknown_pct,total_production_mwh,fossil_free_avg_pct,renewable_avg_pct,hours_covered,year,month,day
0,1777223204564,FR,France,2026-04-26,72.671725,2.329597,3.896878,12.375890,7.875096,0.772257,0.078557,0.0,0.0,0.0,820943.463,99.149186,26.477461,17,2026,4,26
1,1777205541513,FR,France,2026-04-25,73.844472,2.209815,5.084784,7.562436,9.987998,1.077564,0.232930,0.0,0.0,0.0,654326.799,98.689506,24.845033,12,2026,4,25


In [10]:
print("\nPipeline State (el_state):")
print_sql_df("SELECT * FROM state.el_state ORDER BY process_ts DESC limit 10")


Pipeline State (el_state):


,process,process_ts,start_timestamp,end_timestamp,status,record_count,error_message
0,gold,1777223204564,2026-04-26 22:36:49.610194+05:30,2026-04-26 22:37:01.854672+05:30,R,8,None
1,silver,1777223153884,2026-04-26 22:36:06.581631+05:30,2026-04-26 22:36:28.640011+05:30,C,32,None
2,bronze,1777223123493,2026-04-26 22:35:23.493984+05:30,2026-04-26 22:35:36.179015+05:30,C,8,None
3,bronze,1777220910307,2026-04-26 21:58:30.307016+05:30,2026-04-26 21:58:34.763617+05:30,C,2,None
4,bronze,1777220841657,2026-04-26 21:57:21.657825+05:30,2026-04-26 21:57:25.021086+05:30,C,2,None
5,bronze,1777220785217,2026-04-26 21:56:25.217956+05:30,2026-04-26 21:56:30.034957+05:30,C,2,None
6,bronze,1777220614789,2026-04-26 21:53:34.789398+05:30,2026-04-26 21:53:39.220870+05:30,C,2,None
7,gold,1777216802197,2026-04-26 20:50:06.957933+05:30,2026-04-26 20:50:19.564635+05:30,R,8,None
8,silver,1777216202290,2026-04-26 20:40:08.761385+05:30,2026-04-26 20:40:18.820470+05:30,C,16,None
9,bronze,1777215613422,2026-04-26 20:30:13.422801+05:30,2026-04-26 20:30:25.280276+05:30,C,2,None
